In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv('YOLO_CONFIG_DIR'):
    os.environ['YOLO_CONFIG_DIR'] = os.getenv('YOLO_CONFIG_DIR')

os.environ['CUDA_VISIBLE_DEVICES'] = os.getenv('CUDA_VISIBLE_DEVICES', '1')

import torch
from ultralytics import YOLO, settings

# 1. 설정 및 GPU 확인 (검증 타임)
print(f"⚙️ 설정 파일 위치: {settings.file}") 
# -> 결과가 /home/j-i14c109/... 로 나와야 성공! (/tmp 나오면 실패)

print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")

print(f"현재 할당된 GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")

⚙️ 설정 파일 위치: /home/j-i14c109/yolo_config/Ultralytics/settings.json
PyTorch 버전: 2.9.1+cu128
GPU 사용 가능 여부: True
현재 할당된 GPU: NVIDIA L40S


In [ ]:
# 2. 데이터셋 저장 경로도 '내 방'으로 고정 (Permission denied 방지 2차)
current_dir = os.getcwd()
settings.update({'datasets_dir': current_dir})
print(f"📂 데이터셋 저장 위치: {settings['datasets_dir']}")


# 3. 학습 시작
if torch.cuda.is_available():
    print("\n🚀 1번 GPU로 학습을 시작합니다!")
    model = YOLO('yolov8l-pose.pt')

    experiment_name = os.getenv('EXPERIMENT_NAME', 'dog_pose_default')
    
    results = model.train(
        data='dog-pose.yaml', 

        # [학습 시간]
        epochs=100,        # 미세 조정은 시간이 더 걸립니다 (100 -> 150)
        patience=30,       # 참을성도 조금 늘려줍니다
        
        # [핵심 튜닝 1: 해상도 증가] ⭐️
        imgsz=960,         # 640 -> 960 (디테일 인식력 상승)
        batch=4,           # 해상도를 올렸으니 메모리 터짐 방지를 위해 배치는 줄임 (8 -> 4)
        
        # [핵심 튜닝 2: 관절 정확도 우선] ⭐️
        pose=15.0,         # 관절 위치 오차에 대한 가중치 증가 (기본값 12.0 -> 15.0)
        box=5.0,           # 박스 오차 가중치는 조금 낮춤 (기본값 7.5 -> 5.0)
        
        # [데이터 증강 수정]
        degrees=15.0,      # 회전이 너무 심하면 말단 부위가 잘리거나 왜곡됨 (30.0 -> 15.0)
        scale=0.6,         # 스케일 변화 (0.5 -> 0.6)
        fliplr=0.5,        
        mosaic=1.0,        
        
        # [핵심 튜닝 3: 후반부 정밀 학습] ⭐️
        close_mosaic=20,   # 마지막 20 에포크는 Mosaic를 끄고 원본 이미지로만 학습
        
        # [저장 설정]
        project=f'{current_dir}/runs',
        name=experiment_name
    )
    print("✅ 학습 완료!")
else:
    print("❌ GPU가 잡히지 않았습니다. 설정이나 번호를 다시 확인해주세요.")

📂 데이터셋 저장 위치: /home/j-i14c109

🚀 1번 GPU로 학습을 시작합니다!
Ultralytics 8.4.7 🚀 Python-3.10.19 torch-2.9.1+cu128 CUDA:0 (NVIDIA L40S, 45589MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dog-pose.yaml, degrees=15.0, deterministic=True, device=1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dog_pose_tuned_highres2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os

experiment_name = os.getenv('EXPERIMENT_NAME', 'dog_pose_default')

# 1. 방금 학습된 모델 경로 찾기
model_path = f"{os.getcwd()}/runs/{experiment_name}/weights/best.pt"

print(f"📂 모델 로드 중: {model_path}")
model = YOLO(model_path)

# 2. 이미지 추론
img_path = "dog_down.jpg"

if not os.path.exists(img_path):
    print("❌ 파일이 없습니다! 현재 폴더에 강아지 사진을 올려주세요.")
else:
    results = model(img_path)
    img = cv2.imread(img_path)
    
    # 3. 번호 그리기
    for result in results:
        keypoints = result.keypoints.xy.cpu().numpy()
        if len(keypoints) > 0:
            kpts = keypoints[0]
            print(f"▶ 탐지된 관절 개수: {len(kpts)}개")
            
            for i, (x, y) in enumerate(kpts):
                if x == 0 and y == 0: continue
                
                cv2.circle(img, (int(x), int(y)), 3, (255, 255, 255), -1) # 흰 테두리
                cv2.circle(img, (int(x), int(y)), 2, (0, 0, 255), -1)     # 빨간 점
                
                # 검은 테두리 글씨
                cv2.putText(img, str(i), (int(x)+4, int(y)-4), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
                # 노란색 알맹이 글씨
                cv2.putText(img, str(i), (int(x)+4, int(y)-4), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

    # 4. 파일로 저장
    save_path = "check_points.jpg"
    cv2.imwrite(save_path, img)
    print(f"\n✅ 이미지 저장 완료: {save_path}")
    print("👉 좌측 파일 탐색기에서 'check_points.jpg'를 더블클릭해서 확인하세요!")

📂 모델 로드 중: /home/j-i14c109/runs/dog_pose_tuned_highres2/weights/best.pt

image 1/1 /home/j-i14c109/dog_down.jpg: 640x960 1 dog, 6.9ms
Speed: 2.3ms preprocess, 6.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 960)
▶ 탐지된 관절 개수: 24개

✅ 이미지 저장 완료: check_points.jpg
👉 좌측 파일 탐색기에서 'check_points.jpg'를 더블클릭해서 확인하세요!
